In [1]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

import warnings

warnings.filterwarnings("ignore")
import kairo


In [2]:
log = kairo.read_log("../data/reassignment.xes")
#log = kairo.map_columns(log, {
#    "case ID": "case:concept:name", "activity": "concept:name",
#    "timestamp": "time:timestamp", "resource": "org:resource",
#    "event duration": "event:duration_min",
#})


parsing log, completed traces :: 100%|██████████| 9967/9967 [00:00<00:00, 12160.97it/s]


In [3]:
stats = kairo.log_statistics(log)
print(kairo.abstract_log_statistics(stats))


Event log statistics:
cases: 9967, events: 65067, activities: 7, resources: 7
time span: 2020-01-01 01:00:37.100019+00:00 → 2020-12-31 11:50:19.913963+00:00 (526250 minutes)
throughput time (days): min 0.00, mean 0.18, median 0.13, max 2.24
case length (events): min 1, mean 6.5, median 6.0, max 67
activity frequencies: a (19983), b (10016), f (10016), g (10016), d (5020), e (5020), c (4996)
resource frequencies: res_05 (17882), res_07 (13755), res_04 (11886), res_06 (8140), res_03 (5860), res_01 (5706), res_02 (1838)


In [4]:
print(stats.resource_counts)

org:resource
res_05    17882
res_07    13755
res_04    11886
res_06     8140
res_03     5860
res_01     5706
res_02     1838
Name: count, dtype: int64


A resource state is one **calendar window**, not one event: each row describes how the whole
pool behaved in that window.


In [13]:
config = kairo.ResourceConfig(
    features=("events", "active", "duration", "wait", "activity_events", "ho"),
    window_minutes=1440 * 3,                 
    clustering="som", grid=(3, 3), metric="euclidean",
    pca_components=10,
    signal_metric="euclidean",           # windows are vectors, so distance, not divergence
    signal_reference="recent", signal_lookback=3,
)

result = kairo.run_resource(log, config)
result.to_dict()


{'perspective': 'resource',
 'config': {'features': ('events',
   'active',
   'duration',
   'wait',
   'activity_events',
   'ho'),
  'resources': (),
  'activities': (),
  'window_minutes': 4320,
  'skip_pca': False,
  'pca_components': 10,
  'scaling': 'none',
  'clustering': 'som',
  'metric': 'euclidean',
  'grid': (3, 3),
  'som_init': 'random',
  'n_clusters': 6,
  'eps': 0.5,
  'min_samples': 5,
  'signal_metric': 'euclidean',
  'signal_reference': 'recent',
  'signal_lookback': 3,
  'seed': 7},
 'rows': 122,
 'feature_columns': 119,
 'pca_components': 10,
 'n_states': 9,
 'method': 'som',
 'transitions': 77,
 'windows': 122}

In [14]:
kairo.plot_state_grid(result.states).show()
kairo.plot_pca_variance(result.pca).show()

In [15]:
# how far apart the states sit — SOM only, it has a codebook
kairo.plot_state_distances(result.states, config.metric).show()


In [16]:
dist_fig = kairo.plot_state_distribution(result.distribution, result.states)
kairo.add_window_boundaries(dist_fig, result.distribution["window_start"])
dist_fig.show()

signal_fig = kairo.plot_drift_signal(
    result.signal, title="Euclidean distance: window i vs. mean of the 7 windows before i")
signal_fig.show()   # the three injected drifts should stand out


One global trajectory here — the whole pool moves through states together, so there is no
per-case view to pick from.


In [17]:
traj = result.trajectories
fig = kairo.plot_trajectory(
    traj["window_start"], traj["state_id"].to_numpy(), result.states,
    title=f"Resource state per {config.window_minutes}-min window", window_ticks=True)
kairo.add_transition_markers(fig, result.transitions["timestamp"])
fig.show()


In [18]:
# where the signal actually peaks, against the known drift points
kairo.analysis.top_drift_windows(result.signal, k=6)


,window_start,score
0,2020-07-02 01:00:37.100019,230.967186
1,2020-02-24 01:00:37.100019,135.591390
2,2020-01-22 01:00:37.100019,132.630603
3,2020-04-03 01:00:37.100019,126.937632
4,2020-01-19 01:00:37.100019,126.849228
5,2020-07-05 01:00:37.100019,125.072613


In [19]:
profiles = kairo.analysis.state_profiles(result.features, result.states)
print(kairo.abstract_states(result.states, profiles=profiles, max_len=1500))


States from som (3×3 grid), parameters {'grid': (3, 3), 'metric': 'euclidean', 'init': 'random', 'epochs': 5, 'seed': 7}:
- S0: 20 samples (16.4%)
- S1: 26 samples (21.3%)
- S2: 15 samples (12.3%)
- S3: 0 samples (0.0%)
- S4: 0 samples (0.0%)
- S5: 0 samples (0.0%)
- S6: 22 samples (18.0%)
- S7: 21 samples (17.2%)
- S8: 18 samples (14.8%)
Distinguishing features per state (deviation from the overall mean, in stds):
  S0: events:res_07 (+1.4σ), events:res_01 (+1.3σ), active:res_07 (+1.3σ), events:res_03 (+1.3σ), active:res_01 (+1.3σ)
  S1: activity_events:b:res_04 (-1.0σ), activity_events:e:res_06 (-1.0σ), activity_events:d:res_07 (-1.0σ), activity_events:b:res_05 (+1.0σ), ho:res_07→res_05 (+1.0σ)
  S2: active:res_04 (-1.3σ), events:res_04 (-1.3σ), active:res_05 (-1.2σ), events:res_05 (-1.2σ), active:res_06 (-1.1σ)
  S6: active:res_07 (-1.1σ), active:res_03 (-1.1σ), events:res_07 (-1.1σ), active:res_01 (-1.1σ), events:res_03 (-1.1σ)
  S7: duration:res_03 (+1.1σ), ho:res_01→res_07 (-1.1σ

In [20]:
print(kairo.abstract_result(result))

=== resource pipeline run ===

ResourceConfig parameters:
- features: ('events', 'active', 'duration', 'wait', 'activity_events', 'ho')
- resources: ()
- activities: ()
- window_minutes: 4320
- skip_pca: False
- pca_components: 10
- scaling: none
- clustering: som
- metric: euclidean
- grid: (3, 3)
- som_init: random
- n_clusters: 6
- eps: 0.5
- min_samples: 5
- signal_metric: euclidean
- signal_reference: recent
- signal_lookback: 3
- seed: 7

resource feature matrix: 122 rows × 119 columns.
- events (Events per resource): 7 columns
- active (Active cases per resource): 7 columns
- duration (Mean event duration per resource): 7 columns
- wait (Mean wait into resource): 7 columns
- activity_events (Activity-resource event shares): 49 columns
- ho (Handover shares): 42 columns
Column ranges (min → max, mean):
  events:res_01: 7.000 → 107.000, mean 46.770
  events:res_02: 5.000 → 28.000, mean 15.066
  events:res_03: 7.000 → 103.000, mean 48.033
  events:res_04: 40.000 → 182.000, mean 97.

### Narrowing the pool

`resources` and `activities` restrict which columns are produced. Share features keep
log-wide denominators, so a handover share still means the same thing.


In [ ]:
focused = kairo.run_resource(log, kairo.ResourceConfig(
    features=("events", "duration", "ho"),
    resources=("res_01", "res_02"),      # the two the duration drift slowed down
    window_minutes=1440, clustering="kmeans", n_clusters=4,
))
print(focused.features.matrix.shape, focused.features.columns)
kairo.plot_drift_signal(focused.signal, title="res_01 + res_02 only").show()


### Asking a model


In [21]:
# Inspect the prompt without any network call
prompt = kairo.build_prompt("Which resources changed behaviour, and when?",
                            result=result, log=log)
print(prompt)


<knowledge>
The analysis studies state-based concept drift detection in traditional event logs.
A process is monitored through three complementary notions of state:
- Intra-case state: the situation of one running case, derived from its event prefix
  (executed activities, their order, position in the case).
- Resource state: how the resource dimension behaves across all cases in a calendar
  window (workload, handovers, waiting times, activity-resource assignments).
- Inter-case state: the global process situation per calendar window (active cases,
  arrivals, completions, pacing, stalled cases, case-attribute mixes).
Concept drift means the underlying process changed over time. The core idea: compute
states by clustering feature vectors, follow how state occupancy evolves over calendar
windows, and read significant changes in those state signals as drift indicators.
A good analysis names WHERE the signal spikes (which windows), WHAT changed (which
states grew or shrank, which feature

In [22]:
QUESTION = ("You have the details of the resource perspective analysis with the given config. What other parameters would you also use for a new config to get good insights?")


In [ ]:
# Local models
print(kairo.llm.available_models("local"))

In [ ]:
answer = kairo.ask(QUESTION, result=result, log=log, executor=kairo.local_query)
print(answer)


### Hosted providers


In [25]:
answer = kairo.ask(QUESTION, result=result, log=log,
                   executor=kairo.anthropic_query, model="claude-sonnet-5")
print(answer)


## Diagnosis from the current run

Three things in the diagnostics point to configuration problems, not necessarily drift:

**1. Scaling is `none`, but features live on wildly different scales.**
`events:res_05` ranges 43→307 while `activity_events` and `ho` columns are shares in [0,1]. Unscaled PCA is dominated by raw counts — every top loading on PC1–PC8 is an `events:*` or `active:*` column; the 49 activity-share and 42 handover-share columns (91 of 119 columns) barely register. That means the SOM states are effectively just "how busy was each resource," and any real behavioural change (who does what, who hands off to whom) is invisible.
→ **Set `scaling: zscore`** (or similar standardization) so activity-resource and handover shares can compete with volume features.

**2. PCA components are over-specified.**
PC1+PC2 already capture 97.8% of variance (0.899+0.079); PC3–PC10 each contribute ≤0.5%. Keeping `pca_components: 10` just feeds noise dimensions into the SOM.
→ **Let PCA use 

In [ ]:
# Explain a plot
explanation = kairo.explain_plot(
    signal_fig, "What does this drift signal show? Where would you place the drifts?",
    result=result, executor=kairo.anthropic_query, model="claude-sonnet-5")
print(explanation)


In [ ]:
cfg = kairo.nlp_to_config(
    "weekly windows, k-means with 6 clusters, compare each window against the full-log baseline",
    perspective="resource", executor=kairo.anthropic_query,
)
print(cfg)
kairo.run_resource(log, cfg)  # …and run it
